<h1 align="center">Exploratory Data Analysis — Loan Repayment Prediction</h1>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
from scipy.stats import chi2_contingency
from scipy.stats import ks_2samp

# Loading Data

In [ ]:
train = pd.read_csv('../data/train.csv')

test = pd.read_csv('../data/test.csv')

orig = pd.read_csv('../data/original.csv')

## Feature Definitions

In [ ]:
numerical_cols = ['annual_income', 'debt_to_income_ratio', 'credit_score', 
                  'loan_amount', 'interest_rate']


categorical_cols = ['gender', 'marital_status', 'education_level', 
                    'employment_status', 'loan_purpose', 'grade_subgrade']

## Data Inspection

**Shape & columns**

In [ ]:
pd.DataFrame({
    'rows': [train.shape[0], test.shape[0], orig.shape[0]],
    'columns': [train.shape[1], test.shape[1], orig.shape[1]],
}, index=['train', 'test', 'orig'])

- Train dataset has 593,994 rows and 13 columns
- Test dataset has 254,569 rows and 12 columns
- Orig: 20,000 rows, 22 columns — notably more features than synthetic



In [ ]:
print(set(train.columns) - set(test.columns))

- The only difference is `loan_paid_back` — the target variable, which is expected as test set does not contain labels.

In [ ]:
print("Columns in original but not in train:", set(orig.columns) - set(train.columns))
print("Columns in train but not in original:", set(train.columns) - set(orig.columns))

- Original has 10 additional features absent from synthetic: monthly_income, installment, age, loan_term, total_credit_limit, current_balance, num_of_open_accounts, num_of_delinquencies, delinquency_history, public_records — these cannot be used directly but their distributions and target correlations could inform feature engineering ideas for the synthetic data

**Preview**

In [ ]:
for df in [train, test, orig]:
    display(df.head())

**Observations for train dataset:**
- Mix of numerical features: `annual_income`, `debt_to_income_ratio`, `credit_score`, `loan_amount`, `interest_rate`
- Mix of categorical features: `gender`, `marital_status`, `education_level`, `employment_status`, `loan_purpose`, `grade_subgrade`
- Target `loan_paid_back` is binary (0.0 and 1.0)
- `grade_subgrade` is a letter+number code (C3, D3, F1 etc.)

**Observations for test dataset:**
- Same structure as train — columns align as expected

**Observations for original dataset:**
- 20,000 rows, 22 columns — smaller than synthetic but richer in features
- Contains 10 additional features absent from synthetic: `age`, `monthly_income`, `installment`, `loan_term`, `total_credit_limit`, `current_balance`, `num_of_open_accounts`, `num_of_delinquencies`, `delinquency_history`, `public_records`
- These extra features cannot be used directly (not present in test), but their correlations with the target could inform feature engineering ideas
- Shared features appear structurally consistent with synthetic — same categorical values, same numerical scale

**Info and missing values**

In [ ]:
for name, df in [('train', train), ('test', test), ('orig', orig)]:
    print(f"=== {name.upper()} ===")
    df.info()
    print(df.isnull().sum())
    print()

**Observations for train dataset:**
- No missing values across all features
- 5 float64 (annual_income, debt_to_income_ratio, loan_amount, interest_rate, loan_paid_back), 2 int64 (id, credit_score), 6 str (all categoricals)
- loan_paid_back is float64 — casting to int recommended
- Dtype casting for some features is recommended:  loan_paid_back -> int (target variable), grade_subgrade, possibly education_level -> ordinal - possible ordinals should be more thoroughly examined.
- id is int — should be dropped before training, no predictive value


**Observations for test dataset:**
- Same structure as train minus loan_paid_back, no missing values — clean and consistent.

**Observations for original dataset:**
- No missing values across all 22 columns — clean
- `loan_paid_back` is int64 (0/1) vs float64 (0.0/1.0) in synthetic — same information, different dtype
- 10 extra features not present in synthetic: `age`, `monthly_income`, `installment`, `loan_term`, `total_credit_limit`, `current_balance`, `num_of_open_accounts`, `num_of_delinquencies`, `delinquency_history`, `public_records` — cannot be used as direct model features but can serve as a richer target encoding source via shared columns for example


**Duplicates**

In [ ]:
for name, df in [('train', train), ('test', test), ('orig', orig)]:
    print(f"=== {name.upper()} ===")
    print(f"Duplicate rows: {df.duplicated().sum()}")
    if 'id' in df.columns:
        print(f"Duplicate IDs:  {df['id'].duplicated().sum()}")
    print()

    #  Original dataset has no id column — row-level duplicate check only.

- No duplicate rows found in all datasets, they are clean.

In [ ]:
print(f"ID overlap between train and test: {train['id'].isin(test['id']).sum()}")

- No ID leakage between train and test.

# Domain Validation

In [ ]:
impossible_checks = {
    'annual_income < 0':         lambda df: (df['annual_income'] < 0).sum(),
    'loan_amount < 0':           lambda df: (df['loan_amount'] < 0).sum(),
    'credit_score out of range': lambda df: ((df['credit_score'] < 300) | (df['credit_score'] > 850)).sum(),
    'interest_rate out of range':lambda df: ((df['interest_rate'] < 0) | (df['interest_rate'] > 100)).sum(),
    'debt_to_income_ratio < 0':  lambda df: (df['debt_to_income_ratio'] < 0).sum(),
}

unusual_checks = {
    'debt_to_income_ratio > 1 (unusual)':   lambda df: (df['debt_to_income_ratio'] > 1).sum(),
    'debt_to_income_ratio > 2 (very high)': lambda df: (df['debt_to_income_ratio'] > 2).sum(),
}

for name, df in [('train', train), ('test', test), ('original', orig)]:
    print(f"=== {name.upper()} ===")
    print("Impossible values:")
    for label, check in impossible_checks.items():
        print(f"  {label}: {check(df)}")
    print("Unusual values:")
    for label, check in unusual_checks.items():
        print(f"  {label}: {check(df)}")
    print()

- No impossible values detected  — all within realistic ranges

## Target variable analysis

In [ ]:
counts = train['loan_paid_back'].value_counts()

pct = train['loan_paid_back'].value_counts(normalize=True) * 100 # converting to percentage - just for better readability

print(counts)
print()
print(pct.round(1))


**Observation:**

- 474,494 repaid (79.9%) vs 119,500 not repaid (20.1%) — ~4:1 class imbalance - might be needed to be handled

In [ ]:
counts.plot(kind='bar', color=['steelblue', 'salmon'], edgecolor='white', figsize=(6, 4))
plt.xticks([0, 1], ['Repaid (1)', 'Not Repaid (0)'], rotation=0)
plt.title('Target Variable Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

- Visual confirmation of the ~4:1 class imbalance reported in the statistics above.

# Distribution Shift

In [ ]:
print("Numerical columns KS test:")
print()
for col in numerical_cols:
    stat_to, p_to = ks_2samp(train[col], orig[col])
    stat_tt, p_tt = ks_2samp(train[col], test[col])
    print(f"{col}:")
    print(f"  train vs original: KS={stat_to:.4f}, p={p_to:.4f}")
    print(f"  train vs test:     KS={stat_tt:.4f}, p={p_tt:.4f}")

print("\nCategorical columns KS test:")
print()
for col in categorical_cols:
    stat_to, p_to = ks_2samp(
        train[col].astype('category').cat.codes,
        orig[col].astype('category').cat.codes
    )
    stat_tt, p_tt = ks_2samp(
        train[col].astype('category').cat.codes,
        test[col].astype('category').cat.codes
    )
    print(f"{col}:")
    print(f"  train vs original: KS={stat_to:.4f}, p={p_to:.4f}")
    print(f"  train vs test:     KS={stat_tt:.4f}, p={p_tt:.4f}")

**Observations**:

- Train vs test: no drift — all KS statistics under 0.003, all p-values well above 0.05, consistent with earlier findings
- Train vs original: significant drift across all features — all p-values 0.000, confirming the synthetic dataset diverges meaningfully from the original
- `debt_to_income_ratio` has the largest drift (KS=0.286), followed by annual_income (KS=0.170) — the two most skewed features in the synthetic data
- Categorical drift is moderate — `employment_status` (KS=0.108) and `loan_purpose` (KS=0.104) show the most shift
- This suggests the original dataset should be used cautiously as external data — distributions differ enough that models trained on the original may not transfer cleanly to the synthetic train/test

In [ ]:
def plot_dist_three(cols, title, plot_type='hist'):
    n = len(cols)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))

    for i, col in enumerate(cols):
        ax = axes[i]
        if plot_type == 'hist':
            ax.hist(train[col], bins=50, alpha=0.5, color='steelblue', label='train')
            ax.hist(test[col],  bins=50, alpha=0.5, color='salmon',    label='test')
            ax.hist(orig[col],  bins=50, alpha=0.5, color='green',     label='original')
        else:
            all_cats = sorted(set(train[col].unique()) | set(test[col].unique()) | set(orig[col].unique()))
            train_pct = train[col].value_counts(normalize=True).reindex(all_cats, fill_value=0)
            test_pct  = test[col].value_counts(normalize=True).reindex(all_cats, fill_value=0)
            orig_pct  = orig[col].value_counts(normalize=True).reindex(all_cats, fill_value=0)
            x = range(len(all_cats))
            w = 0.25
            ax.bar([i - w for i in x], train_pct, width=w, alpha=0.7, color='steelblue', label='train')
            ax.bar([i     for i in x], test_pct,  width=w, alpha=0.7, color='salmon',    label='test')
            ax.bar([i + w for i in x], orig_pct,  width=w, alpha=0.7, color='green',     label='original')
            ax.set_xticks(list(x))
            ax.set_xticklabels(all_cats, rotation=45, ha='right')

        ax.set_title(col)
        ax.legend()

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_dist_three(numerical_cols, 'Numerical Distribution: Train vs Test vs Original')
plot_dist_three(categorical_cols, 'Categorical Distribution: Train vs Test vs Original', 'bar')

**Observations (Visualization):**
- Train and test overlap almost perfectly across all features — visually confirms the KS results
- Visually confirmed statistically significant drift between train and original
- `credit_score` and `loan_amount`: original has a smoother, more spread distribution while synthetic has sharp peaks and jagged multimodal shape, suggesting the deep learning generator introduced artificial clustering.
- `employment_status`: original has notably more `Self-employed` and less `Employed` dominance than synthetic
- `loan_purpose`: original has far less `Debt consolidation` dominance (~40% vs ~55% in synthetic)
- `grade_subgrade`: most consistent across all three datasets — original and synthetic align closely

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

corr_synthetic = train[numerical_cols].corr(method='spearman')
corr_original  = orig[numerical_cols].corr(method='spearman')

sns.heatmap(corr_synthetic, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Spearman Correlation — Synthetic (Train)')

sns.heatmap(corr_original, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Spearman Correlation — Original')

plt.tight_layout()
plt.show()

**Observation**:

- `credit_score` ↔ `interest_rate`: -0.52 synthetic vs -0.55 original — virtually identical, relationship fully preserved
- All other pairs: near-zero in both — no relationships emerged or disappeared
- Feature interactions are stable across synthetic and original despite distribution shift.
- The synthetic generator preserved inter-feature relationships while altering individual distributions. This slightly strengthens the case for using the original as a feature engineering source — the structure is trustworthy even if the values differ.

In [ ]:
print(f"Train repayment rate:    {train['loan_paid_back'].mean():.3f}")
print(f"Original repayment rate: {orig['loan_paid_back'].mean():.3f}")

- Target distribution is consistent across synthetic and original datasets (0.799 vs 0.800) — no class balance shift introduced by the generator.

<h2 align="center">Final conclusion for test and original dataset</h2>

- Both test and original datasets have been inspected — no missing values, no duplicates, no unrealistic values detected.
- Test is structurally identical to train minus the target variable — inspection complete.
- Original is clean but differs meaningfully in distributions and contains 10 extra features unavailable in synthetic data. Inter-feature relationships are preserved (Spearman correlation stable: -0.52 vs -0.55), strengthening the case for use as a feature engineering or blending source despite distribution shift.

## Univariate Analysis

**Categorical feature inspection**

In [ ]:
for col in train.select_dtypes(include='str').columns:
    print(f"{col}: {train[col].nunique()} unique")

**Observation:**
- No inconsistent formatting — all values are clean and consistently capitalized
- `grade_subgrade`: 30 unique values — high cardinality - suspected ordinal structure - to be determined
- `loan_purpose`: 8 unique values — moderate cardinality
-  All others have low cardinality (3-5 values) — one-hot or ordinal depending on whether natural order exists could be appropriate.

In [ ]:
for col in train.select_dtypes(include='str').columns:
    print(f"\n{col}:")
    print(train[col].value_counts(normalize=True).round(3))

**Observations (Relative Frequency Distribution):**
- `gender`: balanced — Female 51.5%, Male 47.8%, Other 0.6% rare
- `marital_status`: balanced — Single/Married ~95%, Divorced and Widowed rare
- `education_level`: Bachelor's dominant (47.1%), PhD and Other rare
- `employment_status`: strongly imbalanced — Employed 75.9%, Retired and Student rare
- `loan_purpose`: imbalanced — Debt consolidation 54.7%, Vacation and Medical rare
- `grade_subgrade`: C-grade dominates (~46% combined), A-grades extremely rare, 30 categories total
- 3 different types for `gender`, so it stays as str - not binary 
- as expected only two values for `loan_paid_back` - binary
- education level has 4 concrete types + 1 ambigious : `other`


**Numerical feature inspection**

In [ ]:
train.describe().round(2) # 2 decimal places instead of 6 which are hard to read and provide little value

**Observations:**
- `annual_income`:  mean 48k, max 393k, std 26k — high spread, strong right skew likely
- `debt_to_income_ratio`:  median 0.10 but max 0.63 — tight core, wide upper tail
- `credit_score`: mean 681, std 55 — realistic, well-centered
- `loan_amount`:  mean 15k, max 49k, std 6.9k — wide spread, to be examined further
- `interest_rate`: mean 12.36, std 2.01 — tight, realistic distribution, no concerns
- all numerical features may require scaling — they are on different scales


## Skewness 

**Note:** 
With 594k rows the normal statistical test for skewness would detect even trivial deviations from normality, so skewness statistics and visualizations are preffered over it.

In [ ]:
print(train[numerical_cols].skew().round(2)) # on entire dataset, not just sample, to confirm skewness is present in full data

**Observations (skewness):**
- `annual_income`: heavily right skewed 
- `debt_to_income_ratio`: heavily right skewed
- `credit_score`: Relatively Normal (slight left skew)
- `loan_amount`: Relatively Normal (slight right skewed)
- `interest_rate`: Normal
- `annual_income` and `debt_to_income_ratio`  may require transformation depending on model choice.

In [ ]:
fig, axes = plt.subplots(len(numerical_cols), 2, figsize=(14, 20))

for i, col in enumerate(numerical_cols):
    
    # Histogram
    axes[i, 0].hist(train[col], bins=50, color='steelblue', edgecolor='white')
    axes[i, 0].set_title(f'{col} — Histogram')
    
    # QQ Plot
    stats.probplot(train[col], dist="norm", plot=axes[i, 1])
    axes[i, 1].set_title(f'{col} — QQ Plot')

plt.tight_layout()
plt.show()

**Observations (Histogram + QQ Plot):**
- `annual_income`: strong right-skewed visible — QQ upper tail shoots far above the line, extreme outliers confirmed
- `debt_to_income_ratio`: right-skewed, values concentrated near 0 — QQ upper tail deviation confirms non-normality
- `credit_score`: approximately normal (with small multimodality) — QQ nearly straight, slight ceiling effect at 850 (max possible score)
- `loan_amount`: mildly right-skewed with slight bimodality (10k and 20k), lower tail compressed — QQ upper tail curves above line
- `interest_rate`: symmetric — QQ nearly perfect straight line fit

Should be investigated more regarding outliers - especially `annual_income` and `debt_to_income_ratio`

## Kurtosis 

In [ ]:
print(train[numerical_cols].kurtosis().round(2))

**Observations (Kurtosis):**
- `annual_income`: 7.09 — extremely heavy tails, extreme outliers confirmed
- `debt_to_income_ratio`: 2.34 — moderately heavy tails, outliers present but less severe
- `credit_score`: 0.10 — normal tail behavior, no outlier concern
- `loan_amount`: -0.15 — lighter tails than normal, non-normality driven by clustering not outliers
- `interest_rate`: 0.06 — normal tail behavior, no outlier concern

Extreme outliers may require handling depending on model choice.

## Outlier Analysis

In [ ]:
# Some features are heavily right-skewed so IQR method is appropriate.

for col in numerical_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((train[col] < Q1 - 1.5 * IQR) | (train[col] > Q3 + 1.5 * IQR)).sum()
    print(f"{col}: {outliers} outliers ({outliers/len(train)*100:.2f}%)")

**Observations (IQR Outlier Check):**
- `annual_income`: 2.68% outliers but highly extreme — consistent with kurtosis 7.09 and QQ upper tail, warrants attention
- `debt_to_income_ratio`: most outliers (2.96%) but less extreme — more numerous, not as severe
- `credit_score`, `interest_rate`: under 1% — minor, consistent with near-normal distributions
- `loan_amount`: under 1% — non-normality driven by clustering not outliers, confirmed
- Outlier levels are manageable overall, but `annual_income` and `debt_to_income_ratio` warrant attention.

In [ ]:
fig, axes = plt.subplots(1, len(numerical_cols), figsize=(20, 10))

for i, col in enumerate(numerical_cols):
    axes[i].boxplot(train[col], vert=True)
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

- Overall the visualization confirms statistical IQR outlier analysis
- `annual_income` has the most extreme outliers stretching far above the box
- `debt_to_income_ratio` has numerous but less extreme ones
- The remaining three features are largely clean with only minor outliers visible.

## Bivariate analysis ##

In [ ]:
train.groupby('loan_paid_back')[numerical_cols].agg(['mean', 'median']).round(2)

**Observations (Numerical Features vs Target):**
- `credit_score`: strong separator — defaulters score ~32 points lower (655 vs 687)
- `debt_to_income_ratio`: strongest separator — defaulters have 55% higher ratio (0.17 vs 0.11), median confirms (not skew-driven)
- `interest_rate`: modest signal — defaulters pay slightly higher rates
- `annual_income`: very similar medians — weak predictor
- `loan_amount`: near-identical — the weakest predictor

In [ ]:
fig, axes = plt.subplots(1, len(numerical_cols), figsize=(20, 10))

for i, col in enumerate(numerical_cols):
    train.boxplot(column=col, by='loan_paid_back', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('loan_paid_back')

plt.suptitle('Numerical Features vs Target')
plt.tight_layout()
plt.show()

- Visually confirms statistical observations - no surprise

In [ ]:
for col in categorical_cols:
    print(train.groupby(col)['loan_paid_back'].mean().round(3).sort_values())
    print()

**Observations (Categorical Features vs Target):**
- `employment_status`: strongest categorical predictor — Unemployed repay only 7.8%, Retired nearly 99.7%
- `grade_subgrade`: strong ordinal signal — F-grades ~60%, A-grades ~95% repayment
- `loan_purpose`: modest signal — Education/Medical lowest (~77.7%), Home highest (~82.3%)
- `education_level`: weak signal — natural ordinal order not confirmed in data (High School 81.0% > Bachelor's 78.9%), encoding strategy to be determined
- `gender`: no signal — all categories ~79.5-80.2%
- `marital_status`: no signal — all categories ~79.0-79.9%

In [ ]:
pd.crosstab(train['employment_status'], train['loan_paid_back'], normalize='index').round(3)

- Repayment spread confirms extreme separation: Retired 99.7% vs Unemployed 7.8% vs Student 26.4% — employment_status is by far the strongest predictor in the dataset.

In [ ]:
for col in categorical_cols:
    for name, df in [('train', train), ('orig', orig)]:
        ct = pd.crosstab(df[col], df['loan_paid_back'])
        chi2, p, dof, _ = chi2_contingency(ct)
        n = ct.sum().sum()
        cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
        print(f"{col} ({name}): Cramér's V = {cramers_v:.3f}")
    print()

**Observations (Cramér's V — Train vs Original):**
- `employment_status`: 0.657 vs 0.584 — strongest categorical predictor, confirmed genuine in original but inflated by synthetic generator — model may over-rely on it relative to real-world predictive power
- `grade_subgrade`: 0.228 vs 0.195 — strong association, confirmed in original
- `loan_purpose`: 0.026 vs 0.028 — weak signal, consistent
- `education_level`: 0.025 vs 0.023 — weak signal, consistent
- `gender`: 0.007 vs 0.007 — no meaningful association
- `marital_status`: 0.003 vs 0.007 — no meaningful association
- Categorical-target relationships are preserved across synthetic and original — no data leakage detected


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    repay_rate = train.groupby(col)['loan_paid_back'].mean().sort_values()
    repay_rate.plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_ylabel('Repayment Rate')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

- all conclusions visually confirmed

**Grade Subgrade Deep Dive**

In [ ]:
grade_repay = train.groupby('grade_subgrade')['loan_paid_back'].mean().sort_values(ascending=False)

grade_repay.plot(kind='bar', figsize=(14, 5), color='steelblue', edgecolor='white')
plt.title('Repayment Rate by Grade Subgrade')
plt.ylabel('Repayment Rate')
plt.xlabel('Grade Subgrade')
plt.axhline(y=train['loan_paid_back'].mean(), color='red', linestyle='--', label='Overall Mean')
plt.legend()
plt.tight_layout()
plt.show()

**Observations (Grade Subgrade Deep Dive):**
- Ordinal structure confirmed — repayment rate decreases monotonically from A to F grades
- A-grades: ~95% repayment, all well above overall mean (0.80)
- B/C-grades: straddle the mean — C3 is the clear cutoff point where grades drop below average
- D/E/F-grades: all below mean, F-grades bottom out at ~60%
- 35-point spread (60% to 95%) — largest repayment range of any feature, confirming grade_subgrade as strong categorical predictor

## Multivariate analysis

In [ ]:
corr = train[numerical_cols + ['loan_paid_back']].corr(method='spearman') 

# spearman is more robust to outliers and non-normal distributions than pearson, 
# which is appropriate given the skewness and outliers.


plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Spearman Correlation Heatmap')
plt.tight_layout()
plt.show()

**Observations (Spearman Correlation Heatmap):**
- No significant multicollinearity
- `credit_score` ↔ `interest_rate`: -0.52 — strongest feature relationship, moderate but not concerning
- `debt_to_income_ratio` → `loan_paid_back`: -0.30 — strongest numerical predictor of target
- `credit_score` → `loan_paid_back`: 0.24 — second strongest numerical predictor
- `interest_rate` → `loan_paid_back`: -0.13 — weak signal
- `annual_income` and `loan_amount`: near-zero correlation with target — confirmed weak predictors

<h1 align="center">Final Summary Section</h1>

### Train and Test datasets
- no missing values, no duplicates, no unrealistic values and no distribution drift

### Original Dataset:

- Significant distribution shift confirmed across all features (KS up to 0.286) — synthetic generator altered distributions and category proportions while preserving class balance (0.799 vs 0.800)
- Inter-feature correlation structure fully preserved (Spearman -0.52 vs -0.55 for the strongest pair) — strengthens the case for using original as a feature engineering/blending source despite distribution shift.

### Dataset Shape
- 593,994 rows, 13 columns (11 features + 1 target + id)
- 5 numerical features, 6 categorical features, 1 target variable
- `loan_paid_back`: binary target (0.0/1.0)

### Target Variable
- 80/20 class imbalance.

### Key Predictors (Strongest → Weakest)
**Categorical:**
- `employment_status` — strongest signal (Cramér's V: 0.657), 92-point repayment spread
- `grade_subgrade` — strong confirmed ordinal signal (Cramér's V: 0.228), 35-point spread F→A
- `loan_purpose` — weak signal (Cramér's V: 0.026)
- `education_level` — weak signal (Cramér's V: 0.025), ordinal order broken in data
- `gender` & `marital_status` — no meaningful signal, dropping recommended for linear models; trees will ignore them naturally

**Numerical:**
- `debt_to_income_ratio` — strongest numerical predictor (Spearman: -0.30)
- `credit_score` — second strongest (Spearman: 0.24)
- `interest_rate` — weak signal (Spearman: -0.13)
- `annual_income` & `loan_amount` — near-zero correlation with target, weakest predictors

### Distributions & Outliers
**Skewness:**
- `annual_income`: 1.72 — strong right skew
- `debt_to_income_ratio`: 1.41 — strong right skew
- `loan_amount`: 0.21 — mild right skew
- `credit_score`: -0.17 — approximately normal
- `interest_rate`: 0.05 — approximately normal

**Kurtosis:**
- `annual_income`: 7.09 — extremely heavy tails, extreme outliers likely
- `debt_to_income_ratio`: 2.34 — moderately heavy tails
- `loan_amount`: -0.15 — lighter tails, non-normality driven by clustering
- `credit_score` & `interest_rate`: ~0 — normal tail behavior

**IQR Outlier Check:**
- `annual_income`: 2.68% — few but extreme
- `debt_to_income_ratio`: 2.96% — numerous but less extreme
- `loan_amount`, `credit_score`, `interest_rate`: under 1% — minor

### Multicollinearity
- No significant multicollinearity detected.
- `credit_score` and `interest_rate` only share a moderate relationship.

### Preprocessing & Feature Engineering Recommendations
- `id`: dropping recommended (sequential, no predictive value)
- `loan_paid_back`: casting to int recommended (target variable)
- `annual_income` & `debt_to_income_ratio`: log transformation + optional outlier handling recommended for linear models
- `grade_subgrade`: ordinal encoding recommended (confirmed monotonic A→F structure, high cardinality — 30 unique values)
- `education_level`: one-hot encoding recommended (ordinal order broken in data)
- `employment_status`, `loan_purpose`, `gender`, `marital_status`: one-hot encoding recommended if linear model
- All numerical features: scaling recommended for linear models, not needed for tree-based
- Rare categories (Other in gender, Widowed, Divorced): grouping recommended depending on model 
- class imbalance — may require handling (class weights or threshold tuning) depending on model choice. Less critical given AUC as evaluation metric.
- Original dataset: correlation stability confirmed despite distribution shift — recommended for, for example: target encoding on shared features or blending a model trained on original with synthetic is viable but should not be trained only on the original dataset

<h1 align="center">Model Selection</h1>

Given:
- Mixed feature types (numerical + categorical)
- Non-normal distributions with outliers in key features
- Strong non-linear signals in `employment_status` and `grade_subgrade`
- strong right skewness in some features
- AUC as evaluation metric
- Business requirement, which is kaggle competition efficiency

Decision:
- Tree-based model are the most recommended, they are robust to skewness, outliers, handle mixed feature types natively, capture non-linear relationships naturally, do not require scaling and are well-suited for AUC optimization. Tree based models usually also are most efficient in tabular data competition such as this one. The further decision which particularly tree-based models or their ensemble etc must be made based on empricial results in model notebook.
- Linear models (Logistic Regression, SVM) would require scaling, transformation, explicit encoding, and outlier handling — significant preprocessing overhead with no AUC advantage over tree-based models
- Neural networks are overkill for tabular data of this size — harder to tune, no significant AUC advantage over GBMs on structured data